In [1]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.0EgCjqR4Xs/ipykernel_3774692/1270949041.py:4: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration

1. Load all separate datasets (separated by year) and combine into one csv

- This saves out a csv with all warnings together in one file: /home/csutter/DRIVE-clean/snow_squall/data/nws_warnings/nws_all_warnings.csv
- Don't run this again

In [24]:
files = ["/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/StormEvents_details-ftp_v1.0_d2022_c20250721.csv",
"/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/StormEvents_details-ftp_v1.0_d2023_c20260116.csv",
"/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/StormEvents_details-ftp_v1.0_d2024_c20260116.csv",
"/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/StormEvents_details-ftp_v1.0_d2025_c20260116.csv"]

ds = []
lens = []
for x in files:
    d = pd.read_csv(x)
    d = d[d["STATE"]=="NEW YORK"]
    lens.append(len(d))
    ds.append(d)

df = pd.concat(ds)

# Should be this many NY events (rows)
print(sum(lens))
print(len(df))

df.to_csv("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/ncei_ny_events.csv")

7802
7802


In [26]:
ck = df[((df["EVENT_TYPE"]=='Heavy Snow')&(df["YEAR"] == 2025))]
print(len(ck))

2


In [27]:
# df.head(4)

In [21]:
7802*3

23406

2. Add in geometry for NWS regions
- Don't run this again. Keeping for reference for mapping to zones and counties geometries
- This uses the combined csv made from step 1 to add in spatial information
- Note that this dataset, for some events, contains begin_lat, end_lat, begin_lon, end_lon, presumably making a smaller boxed region (subset of the county/zone) for which the event took place

In [23]:
df = pd.read_csv("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/ncei_all_events.csv") 

df = df[df["STATE"]=="NEW YORK"]

print(len(df))

23406


In [28]:
# Load data from step 1
df = pd.read_csv("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/ncei_ny_events.csv") 

print("initial check of events length")
print(len(df))

# df = df[df["STATE"]=="NEW YORK"]

# Load GIS data
# https://www.weather.gov/gis/IDP-GISRestMetadata
# Load zones dataset https://www.weather.gov/gis/publiczones
zones_gdf = gpd.read_file('/home/csutter/DRIVE-clean/NWS_warnings/data/shapefile/z_18mr25/z_18mr25.shp') # Column here is ZONE
# Load counties https://www.weather.gov/gis/Counties 
counties_gdf = gpd.read_file('/home/csutter/DRIVE-clean/NWS_warnings/data/shapefile/c_18mr25') # Column here is FIPS


# Subset the NY locations. This is only critical for the ny_counties one due to the way we create the county code from the FIPS code. However, subsetting the other dfs to work with smaller datasets.
ny_zones = zones_gdf[zones_gdf["STATE"]=="NY"]
ny_counties = counties_gdf[counties_gdf["STATE"]=="NY"]

# Inspect the geometries. Don't want to have multiple CRS (coordinate reference systems) in one dataset
ny_zones = ny_zones[["ZONE","geometry"]]
ny_counties = ny_counties[["FIPS","geometry"]]
print("zones_slice.crs:", ny_zones.crs, "type:", type(ny_zones))
print("counties_slice.crs:", ny_counties.crs, "type:", type(ny_counties))
# different CRS  - convert the county crs to use the zone crs
ny_counties = ny_counties.to_crs(ny_zones.crs)
print("new")
print("counties_slice.crs:", ny_counties.crs, "type:", type(ny_counties))

# Prepare data - format zone and county codes so that we can join the NCEI dataset to the geometries in the county and zone gdfs. See REFERENCE work below to see why this is the mapping that works
# County:
df_county = df[df["CZ_TYPE"]=="C"] # subset to counties since we'll join on the ny_counties gdf
df_county["CZ_FIPS_FORMAT"] = df_county['CZ_FIPS'].astype(str).str.zfill(3) # pad so that we have leading 0s (3 digits total)
ny_counties["FIPS_FORMAT"] = ny_counties['FIPS'].apply(lambda x: x[-3:]) # last 3 digits
events_county = df_county.merge(ny_counties, how = "left", left_on = "CZ_FIPS_FORMAT", right_on = "FIPS_FORMAT") 
print("joined counties")
print(len(df_county))
print(len(events_county))
# Zone:
df_zone = df[df["CZ_TYPE"]=="Z"]
df_zone["CZ_FIPS_FORMAT"] = df_zone['CZ_FIPS'].astype(str).str.zfill(3)
events_zone = df_zone.merge(ny_zones, how = "left", left_on = "CZ_FIPS_FORMAT", right_on = "ZONE") 
print("joined counties")
print(len(df_zone))
print(len(events_zone))

# Check that the combination of these events_zones and events_counties dfs is the same as the original events df (named df)
print(len(df))
print(len(events_zone) + len(events_county))

# If check above is all good, concat the two dfs
events = pd.concat([events_zone,events_county])

# Turn it into a geodataframe
events = gpd.GeoDataFrame(events, geometry="geometry", crs=ny_zones.crs)
print(type(events))



initial check of events length
7802
zones_slice.crs: GEOGCRS["NAD83",DATUM["North American Datum 1983",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ID["EPSG",6269]],PRIMEM["Greenwich",0,ANGLEUNIT["Degree",0.0174532925199433]],CS[ellipsoidal,3],AXIS["longitude",east,ORDER[1],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["latitude",north,ORDER[2],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["ellipsoidal height (h)",up,ORDER[3],LENGTHUNIT["metre",1,ID["EPSG",9001]]]] type: <class 'geopandas.geodataframe.GeoDataFrame'>
counties_slice.crs: epsg:4269 type: <class 'geopandas.geodataframe.GeoDataFrame'>
new
counties_slice.crs: GEOGCRS["NAD83",DATUM["North American Datum 1983",ELLIPSOID["GRS 1980",6378137,298.257222101,LENGTHUNIT["metre",1]],ID["EPSG",6269]],PRIMEM["Greenwich",0,ANGLEUNIT["Degree",0.0174532925199433]],CS[ellipsoidal,3],AXIS["longitude",east,ORDER[1],ANGLEUNIT["Degree",0.0174532925199433]],AXIS["latitude",north,ORDER[2],ANGLEUNIT["Degree",0.0174532925199433]]

/tmp/tmp.0EgCjqR4Xs/ipykernel_3774692/3293549990.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_county["CZ_FIPS_FORMAT"] = df_county['CZ_FIPS'].astype(str).str.zfill(3) # pad so that we have leading 0s (3 digits total)
/tmp/tmp.0EgCjqR4Xs/ipykernel_3774692/3293549990.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_zone["CZ_FIPS_FORMAT"] = df_zone['CZ_FIPS'].astype(str).str.zfill(3)


In [29]:
print(len(df))
print(len(events_zone) + len(events_county))

7802
7802


In [40]:
# FOR REFERENCE IN UNDERSTANDING LOCATIONS (Part 1)

df = df[df["STATE"]=="NEW YORK"]

zc = df[['CZ_TYPE', 'CZ_FIPS', 'CZ_NAME']].drop_duplicates()

display(zc)

print("counties")
print(list(zc[zc["CZ_TYPE"]=="C"]["CZ_FIPS"]))

print("zones")
print(list(zc[zc["CZ_TYPE"]=="Z"]["CZ_FIPS"]))

,CZ_TYPE,CZ_FIPS,CZ_NAME
0,Z,79,NORTHEAST SUFFOLK
2,C,59,NASSAU
3,Z,67,ORANGE
4,Z,68,PUTNAM
5,Z,69,ROCKLAND
...,...,...,...
1306,C,83,RENSSELAER
1307,C,95,SCHOHARIE
1412,C,93,SCHENECTADY
1572,C,47,KINGS


counties
[59, 89, 101, 11, 99, 103, 19, 31, 33, 119, 81, 5, 57, 1, 113, 29, 13, 27, 63, 111, 67, 65, 25, 87, 71, 97, 123, 105, 75, 49, 45, 79, 61, 23, 17, 69, 55, 115, 43, 21, 15, 7, 107, 39, 35, 109, 53, 77, 41, 91, 37, 73, 9, 117, 51, 121, 3, 83, 95, 93, 47, 85]
zones
[79, 67, 68, 69, 70, 177, 176, 71, 78, 80, 81, 179, 35, 34, 26, 29, 30, 87, 27, 28, 31, 66, 65, 40, 32, 33, 39, 42, 82, 43, 83, 48, 178, 38, 49, 50, 52, 51, 75, 53, 54, 61, 73, 74, 41, 47, 59, 64, 84, 8, 6, 19, 20, 9, 72, 37, 18, 36, 17, 46, 21, 60, 13, 85, 4, 63, 3, 14, 7, 1, 10, 11, 2, 12, 5, 58, 22, 25, 55, 56, 57, 62, 24, 23, 45, 44, 16, 15]


In [41]:
# FOR REFERENCE IN UNDERSTANDING LOCATIONS (Part 2)

print("counties gdf")
print(np.unique(ny_counties["FIPS"])) 

# SOLN: The last 3 digits ^ maps to the counties in the NCEI dataset (just need to pad the NCEI county w/ leading 0s)
# EXPLAINATION: The logic: 36 is the state code for New York. The final three digits (059, 089, etc.) are the ANSI county codes. 

counties gdf
['36001' '36003' '36005' '36007' '36009' '36011' '36013' '36015' '36017'
 '36019' '36021' '36023' '36025' '36027' '36029' '36031' '36033' '36035'
 '36037' '36039' '36041' '36043' '36045' '36047' '36049' '36051' '36053'
 '36055' '36057' '36059' '36061' '36063' '36065' '36067' '36069' '36071'
 '36073' '36075' '36077' '36079' '36081' '36083' '36085' '36087' '36089'
 '36091' '36093' '36095' '36097' '36099' '36101' '36103' '36105' '36107'
 '36109' '36111' '36113' '36115' '36117' '36119' '36121' '36123']


In [39]:
# FOR REFERENCE IN UNDERSTANDING LOCATIONS (Part 3)

print("zones gdf")
print(np.unique(ny_zones["ZONE"]))

# SOLN: Just need to pad the NCEI zone code to have leading 0s

zones gdf
['001' '002' '003' '004' '005' '006' '007' '008' '009' '010' '011' '012'
 '013' '014' '015' '016' '017' '018' '019' '020' '021' '022' '023' '024'
 '025' '026' '027' '028' '029' '030' '031' '032' '033' '034' '035' '036'
 '037' '038' '039' '040' '041' '042' '043' '044' '045' '046' '047' '048'
 '049' '050' '051' '052' '053' '054' '055' '056' '057' '058' '059' '060'
 '061' '062' '063' '064' '065' '066' '067' '068' '069' '070' '071' '072'
 '073' '074' '075' '078' '079' '080' '081' '082' '083' '084' '085' '087'
 '176' '177' '178' '179']


In [46]:
print(len(df))

23406


3. Add duration of event in new col

In [32]:
# First need to convert the start and end time cols (which are in EST, always -5, NEVER daylight standard time regardless of when the event occured). there is a col in the df to indiciate that indeed every time is EST -5

# 1. Convert columns from object strings to datetime
# dayfirst=False is standard for US weather data (MM/DD/YY)
events['BEGIN_DATE_TIME'] = pd.to_datetime(events['BEGIN_DATE_TIME'])
events['END_DATE_TIME'] = pd.to_datetime(events['END_DATE_TIME'])

# 2. Extract the numeric offset from the timezone string (e.g., 'EST-5' -> -5)
# This uses a regex to find the plus or minus sign followed by digits
events['offset_val'] = events['CZ_TIMEZONE'].str.extract(r'([+-]?\d+)').astype(int)

# 3. Create UTC columns
# We use pd.to_timedelta to create a time shift. 
# If offset is -5 (5 hours behind UTC), we subtract -5 hours to move "forward" to UTC.
events['BEGIN_UTC'] = events['BEGIN_DATE_TIME'] - pd.to_timedelta(events['offset_val'], unit='h')
events['END_UTC'] = events['END_DATE_TIME'] - pd.to_timedelta(events['offset_val'], unit='h')

# Clean up: remove the temporary offset helper column
events = events.drop(columns=['offset_val'])

In [33]:
# Add duration

events["duration"] = events["END_UTC"] -events["BEGIN_UTC"]
events["duration_sec"] = events["duration"].dt.total_seconds() # geopandas df can't save a timedelta "duration" col so convert it to seconds and drop the timedelta columns
events = events.drop(columns=["duration"])

In [60]:
events.dtypes

Unnamed: 0.2                int64
Unnamed: 0.1              float64
Unnamed: 0                float64
BEGIN_YEARMONTH             int64
BEGIN_DAY                   int64
                        ...      
FIPS                       object
FIPS_FORMAT                object
BEGIN_UTC          datetime64[ns]
END_UTC            datetime64[ns]
duration_sec              float64
Length: 62, dtype: object

4. Save out the final gdf

In [34]:
events.to_file("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/ncei_ny_events_clean.gpkg", driver="GPKG")